# Evaluación de modelos de resumen

**Grupo 2 — Los Predictores** · Proyecto Integrador I · UdeA 2026-2

## 1. Objetivo

Comparar los modelos preentrenados del diseño experimental sobre el problema
planteado —resumir artículos científicos que exceden la ventana de contexto— y
determinar cuáles ofrecen el mejor compromiso entre **calidad** y **costo
computacional**.

Los modelos evaluados son los cuatro del [ADR-003](../docs/adr/003-rol-de-led-y-longt5.md):

| Modelo | Ventana | Papel |
|---|---|---|
| `facebook/bart-large-cnn` | 1.024 | Factorial — objeto de estudio |
| `google/pegasus-arxiv` | 1.024 | Factorial — objeto de estudio |
| `google/long-t5-tglobal-base` | 4.096 | Baseline de referencia |
| `allenai/led-base-16384` | 16.384 | Baseline de referencia |

### Qué NO hace este notebook

No ejecuta los modelos. La inferencia corre en `scripts/run_experiment.py`,
que escribe una fila por documento en `experiments/results/*.jsonl`. Este
notebook **solo lee y analiza** esas filas.

Esa separación es deliberada: el mismo código de evaluación
(`src/resumidor/eval/`) se ejecuta aquí y en el job de GCP, sin duplicarse.
Si el análisis viviera dentro del notebook, las cifras del informe dependerían
de que alguien reejecutara celdas en el orden correcto.

### Control experimental importante

Las cuatro configuraciones usan una **política de generación idéntica**
(`min_new_tokens=150`, `length_penalty=1.0`, `no_repeat_ngram_size=3`), que
sobrescribe la que trae cada checkpoint.

Sin ese control, cada modelo escribe con la longitud que aprendió de su
dominio de afinado. Medido en la semana 6: BART producía 96 tokens de media y
PEGASUS-arxiv 205, frente a los 209 de los abstracts de referencia. El ROUGE
habría medido esa diferencia de calibración de longitud en lugar de la
capacidad de seleccionar contenido.

## 2. Configuración del entorno

In [ ]:
# En Colab, instalar el paquete del proyecto desde el repositorio:
# !pip -q install "git+https://github.com/carlosforeroudea/ps1-g2.git"
#
# En local, el entorno ya lo monta `make setup` (uv sync).

from pathlib import Path

import pandas as pd

from resumidor.eval import calificar, cargar_resultados, resumir_por_configuracion

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

RESULTADOS_DIR = Path("../experiments/results")
print("Directorio de resultados:", RESULTADOS_DIR.resolve())

## 3. Carga de resultados experimentales

Cada fila es un documento procesado por una configuración. Lleva el resumen
generado, el de referencia y las medidas de costo, además de la semilla y el
identificador de configuración que la produjeron — es lo que permite auditar
de dónde sale cada cifra.

In [ ]:
resultados = cargar_resultados(RESULTADOS_DIR)

print(f"{len(resultados)} filas · {resultados['config_id'].nunique()} configuraciones")
print()
print(resultados.groupby(["config_id", "modelo"]).size().rename("documentos").to_frame())

## 4. Validación de los resultados

Antes de medir nada hay que comprobar que las filas son comparables entre sí.
La comparación pareada del [ADR-000](../docs/adr/000-alcance-y-pregunta-de-investigacion.md)
exige que **todas las configuraciones hayan visto exactamente los mismos
documentos**. Si no, los tests compararían documentos distintos en lugar de
modelos distintos.

In [ ]:
# ¿Todas las configuraciones vieron el mismo conjunto de documentos?
docs_por_config = resultados.groupby("config_id")["doc_id"].apply(set)
comun = set.intersection(*docs_por_config)
todos = set.union(*docs_por_config)

print(f"Documentos en común a todas las configuraciones: {len(comun)}")
print(f"Documentos vistos por alguna configuración:      {len(todos)}")

if comun != todos:
    faltantes = {c: sorted(todos - d) for c, d in docs_por_config.items() if d != todos}
    print("\nAVISO — la muestra no es idéntica entre configuraciones:")
    for c, f in faltantes.items():
        print(f"  {c}: le faltan {len(f)} documentos")
    print("\nSe restringe el análisis a los documentos comunes para que el")
    print("pareado sea válido.")
    resultados = resultados[resultados["doc_id"].isin(comun)]
else:
    print("\nOK: muestra idéntica en todas las configuraciones.")

In [ ]:
# Integridad básica: ningún resumen vacío, ninguna referencia ausente.
problemas = {
    "resúmenes vacíos": (resultados["resumen_generado"].str.strip() == "").sum(),
    "referencias ausentes": resultados["resumen_referencia"].isna().sum(),
    "latencias no positivas": (resultados["latencia_s"] <= 0).sum(),
}
for k, v in problemas.items():
    print(f"  {k:26} {v}")

assert sum(problemas.values()) == 0, "hay filas inválidas; revisar antes de medir"
print("\nTodas las filas son válidas.")

## 5. Evaluación con ROUGE

ROUGE-1 mide coincidencia de unigramas, ROUGE-2 de bigramas y ROUGE-Lsum la
subsecuencia común más larga calculada **oración a oración**.

Esa última necesita el texto segmentado en líneas, y los tres formatos que
aparecen aquí son distintos: los abstracts del corpus ya traen saltos, PEGASUS
emite `<n>` como separador y BART devuelve un bloque corrido.
`preparar_para_lsum()` los normaliza; sin eso, ROUGE-Lsum degeneraría en
ROUGE-L y dejaría de ser comparable con la literatura.

In [ ]:
con_rouge = calificar(resultados, con_bertscore=False)

con_rouge.groupby(["config_id", "modelo"])[["rouge1", "rouge2", "rougeLsum"]].mean()

## 6. Evaluación con BERTScore

ROUGE depende de coincidencias léxicas exactas. En resumen **abstractivo** el
modelo reformula: un resumen correcto que use otras palabras es penalizado por
ROUGE aunque el contenido sea el adecuado.

BERTScore compara representaciones contextuales en lugar de cadenas, así que
captura esa equivalencia semántica. Es caro —carga un modelo propio de ~1,4 GB—
por lo que se calcula en un solo lote.

In [ ]:
completo = calificar(resultados, con_bertscore=True)

completo.groupby(["config_id", "modelo"])[
    ["rouge1", "rouge2", "rougeLsum", "bertscore"]
].mean()

## 7. Integración de métricas de calidad y costo

El proyecto no busca el mejor ROUGE: busca el mejor **compromiso** entre
calidad y costo (objetivo específico 6). Por eso las dos familias de métricas
se leen juntas.

La latencia se reporta como **p50 y p95**, no como media: la distribución de
longitudes del corpus tiene una cola derecha muy larga (máximo de 110.531
tokens frente a una mediana de 7.075), y la media quedaría dominada por unos
pocos documentos.

In [ ]:
tabla = resumir_por_configuracion(completo)
tabla

## 8. Resumen de resultados

In [ ]:
# Lectura conjunta: calidad frente a costo, normalizada contra el mejor de cada columna.
comparacion = tabla.reset_index()[
    ["config_id", "modelo", "n", "rouge1", "rouge2", "rougeLsum", "bertscore",
     "latencia_p50", "latencia_p95", "memoria_pico_mb", "tokens_generados"]
].copy()

comparacion["calidad_rel"] = comparacion["rougeLsum"] / comparacion["rougeLsum"].max()
comparacion["costo_rel"] = comparacion["latencia_p50"] / comparacion["latencia_p50"].min()
comparacion["calidad_por_costo"] = comparacion["calidad_rel"] / comparacion["costo_rel"]

comparacion.sort_values("calidad_por_costo", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 5))
for _, f in comparacion.iterrows():
    ax.scatter(f["latencia_p50"], f["rougeLsum"], s=110, zorder=3)
    ax.annotate(f["config_id"], (f["latencia_p50"], f["rougeLsum"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)

ax.set_xlabel("Latencia p50 por documento (s)  —  menos es mejor")
ax.set_ylabel("ROUGE-Lsum  —  más es mejor")
ax.set_title("Compromiso calidad / costo", fontsize=12, fontweight="bold", loc="left")
ax.grid(alpha=0.25, zorder=0)
for lado in ("top", "right"):
    ax.spines[lado].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
# Longitud generada frente a la referencia: verifica que el control funcionó.
longitudes = completo.groupby(["config_id", "modelo"]).agg(
    tokens_generados=("tokens_salida", "mean"),
    documentos=("doc_id", "count"),
)
longitudes["referencia_palabras"] = completo.groupby(["config_id", "modelo"])[
    "resumen_referencia"
].apply(lambda s: s.str.split().str.len().mean())
longitudes

## 9. Conclusiones preliminares

> Esta sección se redacta a partir de las tablas anteriores una vez ejecutado
> el notebook completo. Los puntos a cubrir:
>
> 1. **Qué modelo gana en calidad** y si la diferencia justifica su costo.
> 2. **Cuánto recupera el contexto corto frente al baseline de contexto largo.**
>    Es la forma que toma la tesis del proyecto: *"la estrategia X sobre
>    contexto corto recupera el N % del ROUGE-Lsum de LED a un M % de su
>    latencia"*.
> 3. **Si el control de longitud funcionó** — comparar `tokens_generados`
>    entre configuraciones en la tabla anterior.

### Limitaciones de esta corrida

- **Muestra reducida.** Es una primera ejecución exploratoria, no el
  experimento definitivo de n≥300 que exige el ADR-000. Sirve para verificar
  la canalización y ordenar los modelos, no para concluir.
- **Asimetría de dominio.** `pegasus-arxiv` está afinado en arXiv y
  `bart-large-cnn` en noticias. No existe un BART-arxiv oficial equivalente,
  así que parte de la diferencia entre ambos viene del dominio de afinado y no
  de la estrategia. Decisión consciente del equipo, registrada en
  `experiments/configs/truncation_pegasus.yaml`.
- **Una sola estrategia.** Aquí solo se evalúa truncamiento. Map-reduce y
  extractivo-abstractivo son Fase 3, y son precisamente las que deberían
  recuperar el ~85 % del artículo que el truncamiento descarta.
- **Medido en MPS local.** Las cifras de latencia y memoria no son
  extrapolables a la GPU L4 de GCP; sirven para comparar entre celdas de esta
  misma corrida, no como valores absolutos.